In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
import pickle

In [2]:
###############################
    #ЗАГРУЗКА + ПРЕДОБРАБОТКА ДАННЫХ
###############################

df_kzhn = pd.read_excel("База_соц_эконом_неоднор_регионов.xlsx", sheet_name="КЖН (рейтинг)")
df_kf = pd.read_excel("База_соц_эконом_неоднор_регионов.xlsx", sheet_name="Коэф фондов", skiprows=2)

df_kf = df_kf.iloc[:, [1, 2, 3]]       # берем только Region, Date, KF
df_kf.columns = ["Region", "Date", "KF"]

df_kzhn["Date"] = pd.to_numeric(df_kzhn["Date"], errors="coerce")
df_kzhn["K_PLQ"] = pd.to_numeric(df_kzhn["K_PLQ"], errors="coerce")
df_kzhn = df_kzhn.dropna(subset=["Region", "Date", "K_PLQ"])

df_kf["Region"] = df_kf["Region"].astype(str).str.strip()
df_kf["Date"] = pd.to_numeric(df_kf["Date"], errors="coerce")
df_kf["KF"] = pd.to_numeric(df_kf["KF"], errors="coerce")
df_kf = df_kf.dropna(subset=["Region", "Date", "KF"])

df = df_kzhn.merge(df_kf, on=["Region", "Date"], how="inner")

# Выбираем за какие годы проводим кластеризацию
df_recent = df[df["Date"].isin([2019, 2020, 2021, 2022, 2023, 2024])]
df_year = df_recent.groupby("Region", as_index=False).agg(K_PLQ=("K_PLQ", "mean"), KF=("KF", "mean"))


df_clust = df_year[["Region", "K_PLQ", "KF"]].dropna().copy()

X = df_clust[["K_PLQ", "KF"]].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [3]:
###############################
    #ПОДБОР ЧИСЛА КЛАСТЕРОВ
###############################
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels_k)
    silhouette_scores.append(sil)
    print(f"k={k}: inertia={km.inertia_:.2f}, silhouette={sil:.4f}")

k=2: inertia=99.24, silhouette=0.4225
k=3: inertia=75.53, silhouette=0.4307
k=4: inertia=54.44, silhouette=0.3730
k=5: inertia=41.59, silhouette=0.4021
k=6: inertia=33.38, silhouette=0.3997
k=7: inertia=27.45, silhouette=0.3625
k=8: inertia=24.92, silhouette=0.3559
k=9: inertia=22.30, silhouette=0.3662
k=10: inertia=19.10, silhouette=0.3577


In [4]:
###############################
    #ФИНАЛЬНАЯ КЛАСТЕРИЗАЦИЯ
###############################

optimal_k = 3  # можно поменять, исходя из вывода выше
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_clust["Cluster"] = kmeans.fit_predict(X_scaled)

In [5]:
region_cluster = df_clust[['Region', 'Cluster']].copy()
region_cluster.head()

,Region,Cluster
0,Алтайский край,0
1,Амурская область,1
2,Архангельская область,0
3,Астраханская область,0
4,Белгородская область,2


In [6]:
cluster_dummies = pd.get_dummies(region_cluster['Cluster'],prefix='Cluster').astype(int)

region_cluster = pd.concat([region_cluster, cluster_dummies], axis=1)

region_cluster = region_cluster.drop(columns=['Cluster', 'Cluster_2'])

region_cluster.head()

,Region,Cluster_0,Cluster_1
0,Алтайский край,1,0
1,Амурская область,0,1
2,Архангельская область,1,0
3,Астраханская область,1,0
4,Белгородская область,0,0


In [15]:
pickle_filename = 'region_cluster_dataset.pkl'
with open(pickle_filename, 'wb') as f:
    pickle.dump(region_cluster, f)